In [4]:
import pandas as pd
import numpy as np
from tqdm import tqdm

In [5]:
books = pd.read_csv('books_with_categories.csv')

In [6]:
from transformers import pipeline
classifier = pipeline("text-classification", 
   model="j-hartmann/emotion-english-distilroberta-base",
   top_k=None)
classifier("I love this!")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528676815330982},
  {'label': 'neutral', 'score': 0.005764580797404051},
  {'label': 'anger', 'score': 0.004419777542352676},
  {'label': 'sadness', 'score': 0.0020923905540257692},
  {'label': 'disgust', 'score': 0.0016119878273457289},
  {'label': 'fear', 'score': 0.0004138521908316761}]]

In [7]:
books['description'][0]

'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst the world ha

In [8]:
classifier(books['description'][0])

[[{'label': 'fear', 'score': 0.6548402905464172},
  {'label': 'neutral', 'score': 0.16985228657722473},
  {'label': 'sadness', 'score': 0.11640942841768265},
  {'label': 'surprise', 'score': 0.020700646564364433},
  {'label': 'disgust', 'score': 0.019100714474916458},
  {'label': 'joy', 'score': 0.01516146119683981},
  {'label': 'anger', 'score': 0.003935148008167744}]]

In [9]:
classifier(books['description'][0].split('.'))

[[{'label': 'surprise', 'score': 0.7296025156974792},
  {'label': 'neutral', 'score': 0.14038558304309845},
  {'label': 'fear', 'score': 0.06816234439611435},
  {'label': 'joy', 'score': 0.04794252663850784},
  {'label': 'anger', 'score': 0.009156364016234875},
  {'label': 'disgust', 'score': 0.002628475660458207},
  {'label': 'sadness', 'score': 0.002122160978615284}],
 [{'label': 'neutral', 'score': 0.44937023520469666},
  {'label': 'disgust', 'score': 0.2735917866230011},
  {'label': 'joy', 'score': 0.10908306390047073},
  {'label': 'sadness', 'score': 0.09362737834453583},
  {'label': 'anger', 'score': 0.040478311479091644},
  {'label': 'surprise', 'score': 0.026970185339450836},
  {'label': 'fear', 'score': 0.006879059597849846}],
 [{'label': 'neutral', 'score': 0.646215558052063},
  {'label': 'sadness', 'score': 0.24273370206356049},
  {'label': 'disgust', 'score': 0.04342268779873848},
  {'label': 'surprise', 'score': 0.028300557285547256},
  {'label': 'joy', 'score': 0.01421148

In [10]:
sentence = books['description'][0].split('.')
predictions = classifier(sentence)

In [11]:
sentence[0]

'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives'

In [12]:
predictions[0]

[{'label': 'surprise', 'score': 0.7296025156974792},
 {'label': 'neutral', 'score': 0.14038558304309845},
 {'label': 'fear', 'score': 0.06816234439611435},
 {'label': 'joy', 'score': 0.04794252663850784},
 {'label': 'anger', 'score': 0.009156364016234875},
 {'label': 'disgust', 'score': 0.002628475660458207},
 {'label': 'sadness', 'score': 0.002122160978615284}]

In [13]:
sorted(predictions[0], key=lambda x: x['score'], reverse=True)

[{'label': 'surprise', 'score': 0.7296025156974792},
 {'label': 'neutral', 'score': 0.14038558304309845},
 {'label': 'fear', 'score': 0.06816234439611435},
 {'label': 'joy', 'score': 0.04794252663850784},
 {'label': 'anger', 'score': 0.009156364016234875},
 {'label': 'disgust', 'score': 0.002628475660458207},
 {'label': 'sadness', 'score': 0.002122160978615284}]

In [14]:
emotion_labels = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
isbn=[]
emotion_scores ={label: [] for label in emotion_labels}


def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x['score'], reverse=True)
        for idx, label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[idx]['score'])

    return {label:np.max(scores) for label, scores in per_emotion_scores.items()} 

In [15]:
for i in range(10):
    isbn.append(books['isbn13'][i])
    sentences=books['description'][i].split('.')
    predictions = classifier(sentences)
    max_emotion_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_emotion_scores[label])

In [16]:
emotion_scores

{'anger': [np.float64(0.9671575427055359),
  np.float64(0.9425276517868042),
  np.float64(0.9723207950592041),
  np.float64(0.7326847910881042),
  np.float64(0.8843896985054016),
  np.float64(0.7271748781204224),
  np.float64(0.8725653886795044),
  np.float64(0.8202822804450989),
  np.float64(0.9155240654945374),
  np.float64(0.8603723645210266)],
 'disgust': [np.float64(0.2735917866230011),
  np.float64(0.3193496763706207),
  np.float64(0.11169014126062393),
  np.float64(0.3514839708805084),
  np.float64(0.2726135849952698),
  np.float64(0.27190348505973816),
  np.float64(0.3853578269481659),
  np.float64(0.29216596484184265),
  np.float64(0.2794812321662903),
  np.float64(0.17792679369449615)],
 'fear': [np.float64(0.10908306390047073),
  np.float64(0.1954362690448761),
  np.float64(0.10400666296482086),
  np.float64(0.15072259306907654),
  np.float64(0.12224278599023819),
  np.float64(0.1196221187710762),
  np.float64(0.18529164791107178),
  np.float64(0.2344871163368225),
  np.floa

In [18]:
emotion_labels = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
isbn=[]
emotion_scores ={label: [] for label in emotion_labels}
for i in tqdm(range(len(books))):
    isbn.append(books['isbn13'][i])
    sentences=books['description'][i].split('.')
    predictions = classifier(sentences)
    max_emotion_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_emotion_scores[label])

100%|██████████| 5197/5197 [44:21<00:00,  1.95it/s]  


In [19]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df['isbn13'] = isbn
emotions_df

,anger,disgust,fear,joy,neutral,sadness,surprise,isbn13
0,0.967158,0.273592,0.109083,0.093627,0.064134,0.051363,0.040564,9780002005883
1,0.942528,0.319350,0.195436,0.143348,0.077130,0.051363,0.040564,9780002261982
2,0.972321,0.111690,0.104007,0.078765,0.064134,0.051363,0.040564,9780006178736
3,0.732685,0.351484,0.150723,0.079672,0.064134,0.051363,0.040564,9780006280897
4,0.884390,0.272614,0.122243,0.095043,0.064134,0.051363,0.040564,9780006280934
...,...,...,...,...,...,...,...,...
5192,0.980877,0.305737,0.148208,0.127782,0.043363,0.030656,0.009569,9788172235222
5193,0.883198,0.338892,0.227765,0.078765,0.064134,0.051363,0.040564,9788173031014
5194,0.947779,0.339218,0.141734,0.066685,0.057625,0.009929,0.009055,9788179921623
5195,0.951104,0.368110,0.214133,0.078765,0.064134,0.051363,0.040564,9788185300535


In [20]:
books = pd.merge(books, emotions_df, on="isbn13")

In [21]:
books.to_csv('books_with_emotions.csv', index=False)